## A joke generation sequential workflow

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver # stores in RAM

In [ ]:
load_dotenv()

llm = ChatGroq(model="moonshotai/kimi-k2-instruct-0905")

In [ ]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [ ]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [ ]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [ ]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

In [ ]:
# Final state value
workflow.get_state(config1)

In [ ]:
# Intermediate state values
list(workflow.get_state_history(config1))

In [ ]:
# Another conversation
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

In [ ]:
workflow.get_state(config2)

In [ ]:
list(workflow.get_state_history(config2))

## Time Travel

- We can get to any checkpoint and execute the workflow from there
- helps in debugging

In [ ]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1204f1-ac3b-65c5-8000-17ad6f99c48a"}})

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1204f1-ac3b-65c5-8000-17ad6f99c48a"}})

In [ ]:
list(workflow.get_state_history(config2))

### Updating State

- we can update any state at any particular checkpoint

In [ ]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1204f1-ac3b-65c5-8000-17ad6f99c48a", "checkpoint_ns": ""}}, {'topic':'samosa'})

In [ ]:
list(workflow.get_state_history(config2))

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1204f9-b5e1-6a7c-8001-6fda4abf2478"}})

In [ ]:
list(workflow.get_state_history(config1))